UTS Data Science

Nama : Novika Ardiyaningtyas

*   Nama : Novika Ardiyaningtyas
*   NIM : 250401020135
*   Kelas: IF401

In [8]:
#STEP 0 — Load & Eksplorasi Awal
import pandas as pd
import numpy as np

# Load Dataset Penguins
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

df = pd.read_csv(url)

print("Shape awal:", df.shape)

# Backup data asli
original_df = df.copy()

print("\nInfo Dataset:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistik Deskriptif:")
print(df.describe())

print("\nPreview Data:")
print(df.head())

Shape awal: (344, 7)

Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB
None

Missing Values:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

Statistik Deskriptif:
       bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
count      342.000000     342.000000         342.000000   342.000000
mean       

In [9]:
#STEP 1 — Hapus Duplikat
print("Jumlah duplikat sebelum:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Jumlah duplikat sesudah:", df.duplicated().sum())

print("Shape setelah hapus duplikat:", df.shape)

Jumlah duplikat sebelum: 0
Jumlah duplikat sesudah: 0
Shape setelah hapus duplikat: (344, 7)


Pada tahap ini, data duplikat diperiksa dan dihapus. Ini dilakukan untuk memastikan bahwa setiap baris data adalah unik, sehingga tidak ada informasi yang terulang yang dapat memengaruhi hasil analisis. Dataset menjadi lebih bersih dan konsisten setelah melewati prosedur ini.

In [10]:
#STEP 2 — Normalisasi String
# Simulasi data tidak konsisten

df.loc[0, 'species'] = ' adelie '
df.loc[1, 'species'] = 'ADELIE'
df.loc[2, 'species'] = ' Adelie'

species_before = df['species'].copy()

print("=== Sebelum Normalisasi Species ===")
print(species_before.value_counts())

# Normalisasi
df['species'] = df['species'].str.strip().str.title()

print("\n=== Sesudah Normalisasi Species ===")
print(df['species'].value_counts())

perubahan_species = (species_before != df['species']).sum()

print("\nJumlah perubahan species:", perubahan_species)

# Simulasi data tidak konsisten

df.loc[3, 'sex'] = ' male '
df.loc[4, 'sex'] = 'FEMALE'

sex_before = df['sex'].copy()

print("\n=== Sebelum Normalisasi Sex ===")
print(sex_before.value_counts(dropna=False))

df['sex'] = df['sex'].astype(str).str.strip().str.title()

print("\n=== Sesudah Normalisasi Sex ===")
print(df['sex'].value_counts(dropna=False))

perubahan_sex = (sex_before != df['sex']).sum()

print("\nJumlah perubahan sex:", perubahan_sex)

=== Sebelum Normalisasi Species ===
species
Adelie       149
Gentoo       124
Chinstrap     68
 adelie        1
ADELIE         1
 Adelie        1
Name: count, dtype: int64

=== Sesudah Normalisasi Species ===
species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

Jumlah perubahan species: 3

=== Sebelum Normalisasi Sex ===
sex
MALE      168
FEMALE    165
NaN        10
 male       1
Name: count, dtype: int64

=== Sesudah Normalisasi Sex ===
sex
Male      169
Female    165
Nan        10
Name: count, dtype: int64

Jumlah perubahan sex: 344


Nilai pada kolom kategorikal seperti species dan sex diseragamkan format penulisannya pada tahap normalisasi string dengan menghapus spasi berlebihan dan menyamakan huruf besar-kecil. Hasilnya, kategori yang sebelumnya ditulis dengan cara yang berbeda tetapi memiliki arti yang sama dapat digabungkan menjadi satu yang konsisten.

In [11]:
#STEP 3 — Imputasi Missing Values
print("Missing sebelum:")
print(df.isnull().sum())

# Numerik → Median
df['bill_length_mm'] = df['bill_length_mm'].fillna(
    df['bill_length_mm'].median())

df['bill_depth_mm'] = df['bill_depth_mm'].fillna(
    df['bill_depth_mm'].median())

df['flipper_length_mm'] = df['flipper_length_mm'].fillna(
    df['flipper_length_mm'].median())

df['body_mass_g'] = df['body_mass_g'].fillna(
    df['body_mass_g'].median())

# Kategorik → Modus
df['sex'] = df['sex'].replace('Nan', np.nan)

df['sex'] = df['sex'].fillna(
    df['sex'].mode()[0])

print("\nMissing sesudah:")
print(df.isnull().sum())

Missing sebelum:
species              0
island               0
bill_length_mm       2
bill_depth_mm        2
flipper_length_mm    2
body_mass_g          2
sex                  0
dtype: int64

Missing sesudah:
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


Pada tahap ini dilakukan penanganan data yang hilang (missing values). Kolom numerik diisi menggunakan nilai median, sedangkan kolom kategorikal diisi menggunakan modus. Setelah imputasi, tidak ada lagi nilai kosong pada dataset sehingga data siap digunakan untuk analisis atau pemodelan tanpa kehilangan banyak informasi.

In [13]:
# STEP 4 — Tangani Outlier (IQR)

for col in ['bill_length_mm', 'bill_depth_mm',
            'flipper_length_mm', 'body_mass_g']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_before = ((df[col] < lower) |
                      (df[col] > upper)).sum()

    print(f"\n=== {col} ===")
    print("Nilai maksimum sebelum :", df[col].max())
    print("Outlier sebelum :", outlier_before)

    df[col] = df[col].clip(lower, upper)

    outlier_after = ((df[col] < lower) |
                     (df[col] > upper)).sum()

    print("Nilai maksimum sesudah :", df[col].max())
    print("Outlier sesudah :", outlier_after)


=== bill_length_mm ===
Nilai maksimum sebelum : 59.6
Outlier sebelum : 0
Nilai maksimum sesudah : 59.6
Outlier sesudah : 0

=== bill_depth_mm ===
Nilai maksimum sebelum : 21.5
Outlier sebelum : 0
Nilai maksimum sesudah : 21.5
Outlier sesudah : 0

=== flipper_length_mm ===
Nilai maksimum sebelum : 231.0
Outlier sebelum : 0
Nilai maksimum sesudah : 231.0
Outlier sesudah : 0

=== body_mass_g ===
Nilai maksimum sebelum : 6300.0
Outlier sebelum : 0
Nilai maksimum sesudah : 6300.0
Outlier sesudah : 0


dataset Penguins memang hampir tidak memiliki outlier yang signifikan setelah missing value diisi. Jadi walaupun kode IQR dijalankan, nilai minimum dan maksimum tidak banyak berubah.

In [14]:
#STEP 5 — Validasi & Eksplorasi Akhir

# Cek missing value
print("Total Missing Value:")
print(df.isnull().sum())

# Cek duplikat
print("\nJumlah Duplikat:")
print(df.duplicated().sum())

# Ukuran dataset akhir
print("\nShape Akhir:")
print(df.shape)

# Nama kolom
print("\nNama Kolom:")
print(df.columns.tolist())

# Info dataset
print("\nInfo Dataset:")
print(df.info())

# Statistik deskriptif
print("\nStatistik Deskriptif:")
print(df.describe())

# Preview data
print("\n10 Data Pertama:")
print(df.head(10))

Total Missing Value:
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64

Jumlah Duplikat:
0

Shape Akhir:
(344, 7)

Nama Kolom:
['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']

Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     344 non-null    float64
 3   bill_depth_mm      344 non-null    float64
 4   flipper_length_mm  344 non-null    float64
 5   body_mass_g        344 non-null    float64
 6   sex                344 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB
None

Statistik Deskriptif:
       bill_lengt

Setelah proses cleaning, dataset tidak memiliki missing value maupun data duplikat. Struktur data menjadi lebih konsisten dan siap digunakan untuk analisis atau pemodelan lebih lanjut.

In [15]:
# STEP 6 — Simpan Dataset Bersih

df.to_csv("penguins_clean.csv", index=False)

print("Dataset berhasil disimpan sebagai 'penguins_clean.csv'")

Dataset berhasil disimpan sebagai 'penguins_clean.csv'


Dataset yang telah dibersihkan berhasil disimpan ke dalam file CSV sehingga dapat digunakan kembali tanpa perlu mengulangi proses pembersihan data.

In [16]:
#STEP 7 — Mengakses API dan Membuat DataFrame
import requests
import pandas as pd

# STEP 7 — Akses API

url_api = "https://jsonplaceholder.typicode.com/users"

response = requests.get(url_api)

if response.status_code == 200:

    api_df = pd.DataFrame(response.json())

    print("Shape Data API:", api_df.shape)

    print("\nKolom API:")
    print(api_df.columns.tolist())

    print("\n5 Data Pertama:")
    print(api_df.head())

else:
    print("Gagal mengambil data. Status:", response.status_code)

Shape Data API: (10, 8)

Kolom API:
['id', 'name', 'username', 'email', 'address', 'phone', 'website', 'company']

5 Data Pertama:
   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \
0  {'street': 'Kulas Light', 'suite': 'Apt. 556',...  1-770-736-8031 x56442   
1  {'street': 'Victor Plains', 'suite': 'Suite 87...    010-692-6593 x09125   
2  {'street': 'Douglas Extension', 'suite': 'Suit...         1-463-123-4447   
3  {'street': 'Hoeger Mall', 'suite': 'Apt. 692',...      493-170-9623 x156   
4  {'street': 'Skiles Walks', 'suite': 'Suite 351...          (254)954-1289   

   

Data berhasil diambil dari API JSONPlaceholder dan dikonversi menjadi DataFrame Pandas. Langkah ini menunjukkan bahwa Pandas tidak hanya dapat membaca file lokal, tetapi juga dapat mengolah data yang diperoleh langsung dari layanan web/API.

Kesimpulan: Pada tahap eksplorasi data digunakan pd.read_csv(), head(), info(), describe(), dan isnull().sum() untuk membaca dataset serta melihat struktur dan kondisi data. Pada tahap penghapusan data duplikat digunakan duplicated() dan drop_duplicates(). Normalisasi data teks dilakukan dengan str.strip(), str.title(), dan str.lower() untuk menyeragamkan penulisan data kategorikal. Penanganan missing value menggunakan fillna() dengan bantuan median() dan mode(). Untuk menangani outlier digunakan metode IQR melalui sintaks quantile() dan clip(). Selanjutnya dilakukan validasi data menggunakan assert, serta eksplorasi akhir dengan columns.tolist() dan shape. Dataset yang telah bersih kemudian disimpan menggunakan to_csv(). Pada tahap terakhir dipelajari cara mengambil data dari API menggunakan requests.get() dan mengubah data JSON menjadi DataFrame menggunakan pd.DataFrame(). Secara keseluruhan, langkah 1–7 mencakup proses eksplorasi data, pembersihan data, validasi, penyimpanan data, dan pengambilan data dari API.